# Bài 08: Đồ án cuối khóa - Phân loại Giao dịch Gian lận

**Bối cảnh:** Bạn là một Nhà khoa học dữ liệu tại một công ty công nghệ tài chính (Fintech). Đội ngũ phòng chống rủi ro đã nhận thấy sự gia tăng của các giao dịch gian lận (fraudulent transactions) trong thời gian gần đây. Việc bỏ sót một giao dịch gian lận có thể gây thiệt hại tài chính lớn cho công ty và làm giảm uy tín. Ngược lại, việc chặn nhầm một giao dịch hợp lệ của khách hàng sẽ gây ra trải nghiệm tồi tệ. 

**Nhiệm vụ của bạn:** Xây dựng một mô hình học máy có khả năng phát hiện các giao dịch gian lận một cách hiệu quả. Bạn cần áp dụng các kiến thức đã học trong suốt khóa học để xử lý vấn đề mất cân bằng dữ liệu nghiêm trọng trong bài toán này.

**Mục tiêu:**
1.  Xây dựng được một quy trình làm việc (workflow) hoàn chỉnh từ khám phá dữ liệu, tiền xử lý, huấn luyện đến đánh giá mô hình.
2.  So sánh hiệu quả của các phương pháp xử lý dữ liệu mất cân bằng khác nhau (ví dụ: SMOTE vs. Cost-Sensitive Learning).
3.  Lựa chọn được mô hình cuối cùng dựa trên một chỉ số đánh giá phù hợp với bài toán.
4.  Trình bày và diễn giải kết quả một cách rõ ràng.

## 1. Dữ liệu

Chúng ta sẽ sử dụng một phiên bản đơn giản hóa của bộ dữ liệu "Credit Card Fraud Detection" nổi tiếng từ Kaggle. Bộ dữ liệu chứa các giao dịch được thực hiện bởi chủ thẻ tín dụng châu Âu vào tháng 9 năm 2013. 

**Các cột dữ liệu:**
- `Time`: Giây trôi qua giữa giao dịch này và giao dịch đầu tiên trong tập dữ liệu.
- `V1`, `V2`, ..., `V28`: Các đặc trưng ẩn danh, là kết quả của phép biến đổi PCA (Principal Component Analysis) để bảo vệ danh tính người dùng. Bạn không cần quan tâm ý nghĩa của chúng.
- `Amount`: Số tiền của giao dịch.
- `Class`: Biến mục tiêu của chúng ta. **1** nếu là giao dịch gian lận, **0** nếu không.

Do tính chất của bài toán, dữ liệu sẽ **cực kỳ mất cân bằng**.

In [ ]:
import pandas as pd

# Tải dữ liệu (giả sử file 'creditcard.csv' nằm cùng thư mục)
# Lưu ý: Bộ dữ liệu gốc rất lớn. Để tăng tốc, chúng ta có thể làm việc với một mẫu của nó.
# Trong khuôn khổ bài tập này, chúng ta sẽ giả lập việc đọc dữ liệu.
# Để chạy thực tế, bạn cần tải bộ dữ liệu từ: https://www.kaggle.com/mlg-ulb/creditcardfraud

try:
    df = pd.read_csv('creditcard.csv')
    # Lấy một mẫu nhỏ hơn để chạy nhanh hơn (ví dụ 50%)
    df = df.sample(frac=0.5, random_state=42)
except FileNotFoundError:
    print("Không tìm thấy file 'creditcard.csv'. Vui lòng tải về từ Kaggle.")
    print("Tạo dữ liệu giả lập để tiếp tục bài tập...")
    from sklearn.datasets import make_classification
    X, y = make_classification(n_samples=50000, n_features=30, weights=[0.998, 0.002], 
                               n_informative=15, n_redundant=5, n_clusters_per_class=1,
                               flip_y=0.05, random_state=42)
    # Đặt tên cột cho giống dữ liệu thật
    cols = ['Time'] + [f'V{i}' for i in range(1, 29)] + ['Amount']
    df = pd.DataFrame(X, columns=cols)
    df['Class'] = y

df.head()

## 2. Các bước thực hiện gợi ý

Đây là một quy trình gợi ý. Bạn được khuyến khích thử nghiệm và sáng tạo thêm.

### Bước 1: Khám phá Dữ liệu (Exploratory Data Analysis - EDA)

1.  Kiểm tra thông tin cơ bản của dữ liệu (`.info()`, `.describe()`).
2.  **Kiểm tra mức độ mất cân bằng:** Đếm số lượng mẫu cho mỗi lớp trong cột `Class` và tính tỷ lệ phần trăm. Trực quan hóa bằng biểu đồ `countplot`.
3.  Phân tích sự khác biệt về `Amount` và `Time` giữa hai lớp. Có phải các giao dịch gian lận thường có số tiền nhỏ/lớn hơn không? Chúng xảy ra vào những thời điểm nào?
4.  **Quan trọng:** Dữ liệu có cần được chuẩn hóa (scaling) không? Nhìn vào cột `Amount` và `Time` so với các cột `V1-V28`.

### Bước 2: Tiền xử lý và Chuẩn bị Dữ liệu

1.  **Chuẩn hóa dữ liệu:** Áp dụng `StandardScaler` hoặc `RobustScaler` cho các cột có thang đo khác biệt (như `Time` và `Amount`). `RobustScaler` thường được ưu tiên hơn trong các bài toán có outlier (điểm ngoại lai), vốn phổ biến trong phát hiện gian lận.
2.  Tách biến độc lập `X` và biến mục tiêu `y`.
3.  **Chia dữ liệu:** Chia `X` và `y` thành tập huấn luyện và tập kiểm tra (`train_test_split`). Hãy nhớ sử dụng `stratify=y` để đảm bảo tỷ lệ mất cân bằng được giữ nguyên trong cả hai tập.

### Bước 3: Lựa chọn Chỉ số Đánh giá

Dựa trên bối cảnh bài toán (thiệt hại do FN lớn), hãy suy nghĩ và lựa chọn một hoặc một vài chỉ số chính để so sánh các mô hình. 

- `Accuracy` có phải là lựa chọn tốt không? Tại sao?
- Giữa `Precision` và `Recall` của lớp 1, bạn sẽ ưu tiên chỉ số nào hơn?
- Bạn sẽ chọn `F1-score`, `F2-score` hay một chỉ số nào khác (ví dụ: `Balanced Accuracy`, `PR AUC`) làm thước đo cuối cùng? Hãy viết ra lựa chọn và lý do của bạn.

### Bước 4: Xây dựng và So sánh các Mô hình

Hãy xây dựng ít nhất 3 mô hình sau để so sánh:

1.  **Mô hình Baseline:** Huấn luyện một mô hình `LogisticRegression` đơn giản trên tập huấn luyện gốc (chưa qua xử lý mất cân bằng). Đánh giá kết quả trên tập test. Đây sẽ là cơ sở để bạn so sánh.

2.  **Mô hình với Cost-Sensitive Learning:** Huấn luyện một mô hình `LogisticRegression` với tham số `class_weight='balanced'`. Đánh giá kết quả.

3.  **Mô hình với Resampling:**
    - Áp dụng `SMOTE` trên tập huấn luyện (`X_train`, `y_train`) để tạo ra `X_train_smote`, `y_train_smote`.
    - Huấn luyện một mô hình `LogisticRegression` trên tập dữ liệu đã qua SMOTE.
    - Đánh giá kết quả trên tập test **gốc** (`X_test`, `y_test`).

**Yêu cầu:** Với mỗi mô hình, hãy in ra `classification_report` và chỉ số chính bạn đã chọn ở Bước 3.

### Bước 5: Phân tích và Lựa chọn Mô hình cuối cùng

1.  Tạo một bảng hoặc DataFrame để tóm tắt kết quả của cả 3 mô hình theo các chỉ số quan trọng (Precision, Recall, F-beta, Balanced Accuracy, ...).
2.  So sánh các mô hình: 
    - Mô hình nào cho `Recall` lớp 1 cao nhất?
    - Mô hình nào có sự cân bằng tốt nhất giữa Precision và Recall (dựa trên chỉ số F-beta bạn chọn)?
    - `SMOTE` và `class_weight` có hiệu quả hơn mô hình baseline không? Phương pháp nào tốt hơn trong trường hợp này?
3.  Vẽ đường cong Precision-Recall (PR Curve) cho cả 3 mô hình trên cùng một biểu đồ. Phân tích và so sánh diện tích dưới đường cong (PR AUC).
4.  **Kết luận:** Dựa trên tất cả các phân tích, hãy chọn ra mô hình mà bạn cho là tốt nhất và giải thích lý do tại sao bạn chọn nó.

### Thử thách thêm (Tùy chọn)

- Thử nghiệm với các thuật toán khác mạnh mẽ hơn như `RandomForestClassifier` hoặc `XGBClassifier`.
- Sử dụng `GridSearchCV` để tìm bộ tham số `class_weight` tùy chỉnh tốt nhất.
- Thử nghiệm các kỹ thuật resampling kết hợp như `SMOTEENN` hoặc `SMOTETomek`.

---

## Bắt đầu thực hiện!

Bây giờ, hãy bắt đầu viết code của bạn trong các cell bên dưới. Chúc bạn may mắn!

In [ ]:
# Viết code của bạn ở đây
# Gợi ý: Chia mỗi bước thành các cell riêng biệt để dễ quản lý và trình bày.